# 04 · Selección de modelos: medimos antes de elegir

Cada modelo de DiputadoDetector se eligió con un experimento pequeño pero cuantitativo. Siempre que es
posible usamos una **verdad de referencia automática**:

| Decisión | Candidatos | Métrica | Verdad de referencia |
|---|---|---|---|
| Transcripción | Whisper base (CPU) vs large-v3-turbo (GPU) | WER y RTF | Transcripciones de MLS (con y sin canal telefónico) |
| Lectura visual | Qwen2.5-VL 7B vs 3B | CER, enlaces exactos y latencia | El texto exacto con el que **nosotros** renderizamos cada imagen |
| Búsqueda de campañas | CLIP ViT-B/32 vs SigLIP2 vs e5 | Recall@1, MRR | Variantes re-renderizadas de cada campaña |
| Voz de salida | MMS-TTS vs Bark | Inteligibilidad (WER ida y vuelta con Whisper) y RTF | El texto narrado |
| Infografía | SDXL-Turbo con 1, 2 y 4 pasos | Puntuación SigLIP2 y latencia | El prompt |
| Razonamiento | Qwen3 8B vs Qwen2.5 7B | F1 y latencia en el test manual | Etiquetas manuales |

Además, calibramos aquí los **rangos de similitud** de la búsqueda de campañas que usa la aplicación.

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from diputado import config

config.ensure_dirs()
config.ensure_ffmpeg_on_path()

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image

from diputado.ai import ollama_server
from diputado.ai.gpu import DEVICE, HAS_CUDA, gpu

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
ollama_server.ensure_server(install_if_missing=False)
RESULTS = {}


def norm(s: str) -> str:
    import re
    import unicodedata
    s = unicodedata.normalize("NFKC", s.lower())
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

## 1. Transcripción: tamaño de Whisper

20 clips del split `dev` de MLS en español, en dos condiciones: audio original y tras el canal
telefónico del notebook 01. **RTF** (*real-time factor*) = tiempo de proceso / duración del audio.

In [ ]:
from transformers import pipeline

from diputado.data import telephony, voices

clips = voices.real_clips("dev", n=20, seed=7)
rng = np.random.default_rng(0)
audios = {"original": [voices.load_wav(p) for p in clips["path"]]}
audios["teléfono"] = [telephony.phone_channel(a, 16000, rng, out_sr=16000) for a in audios["original"]]
refs = [norm(t) for t in clips["transcript" if "transcript" in clips else "texto"]]

asr_rows = []
for key, dev in [("asr_cpu", "cpu"), ("asr", DEVICE)]:
    mid = config.HF_MODELS[key]["id"]
    with gpu.claim("whisper_nb"):
        pipe = pipeline("automatic-speech-recognition", model=mid, device=dev,
                        torch_dtype=torch.float16 if dev == "cuda" else torch.float32)
        pipe({"raw": audios["original"][0], "sampling_rate": 16000}, generate_kwargs={"language": "spanish"})  # calentamiento
        for cond, auds in audios.items():
            t0 = time.perf_counter()
            hyps = [norm(pipe({"raw": a, "sampling_rate": 16000}, generate_kwargs={"language": "spanish", "task": "transcribe"})["text"]) for a in auds]
            el = time.perf_counter() - t0
            dur = sum(len(a) for a in auds) / 16000
            asr_rows.append({"modelo": mid.split("/")[-1], "dispositivo": dev, "audio": cond,
                             "WER": jiwer.wer(refs, hyps), "RTF": el / dur})
    del pipe
    if HAS_CUDA:
        torch.cuda.empty_cache()
asr = pd.DataFrame(asr_rows)
display(asr.pivot_table(index=["modelo", "dispositivo"], columns="audio", values=["WER", "RTF"]).round(3))
RESULTS["asr"] = asr.round(4).to_dict(orient="records")

In [ ]:
w = asr.groupby("modelo")[["WER", "RTF"]].mean()
wb, wt = w.loc["whisper-base"], w.loc["whisper-large-v3-turbo"]
display(Markdown(
    f"**Decisión.** `whisper-large-v3-turbo` en GPU: WER medio {wt.WER:.2f} frente a {wb.WER:.2f} de `whisper-base` "
    f"({(1 - wt.WER / wb.WER) * 100:.0f} % menos errores) y {1 / wt.RTF:.1f} veces más rápido que el tiempo real "
    f"(RTF {wt.RTF:.2f}). El canal telefónico apenas cambia el WER de ninguno de los dos. `whisper-base` en CPU "
    f"(RTF {wb.RTF:.2f}) queda como modo ligero para equipos sin GPU: transcribe peor, pero sigue funcionando. "
    "El WER absoluto incluye diferencias de normalización con las transcripciones de MLS (números escritos con "
    "letras, nombres propios), por eso lo usamos para **comparar** modelos, no como cifra absoluta."))

## 2. Lectura visual: Qwen2.5-VL 7B frente a 3B

Como renderizamos nosotros las capturas, conocemos el texto exacto. Medimos el **CER** (tasa de error de
caracteres) del `texto_completo` extraído, si el **enlace** sale copiado exactamente (un error en un
carácter cambia el dominio) y la latencia en caliente.

In [ ]:
from diputado.ai import vision
from diputado.core import campaigns, extractors
from diputado.data import demo_cases

import re

gt = [(config.CAMPAIGNS_DIR / "img" / f"{c['id']}.png", c["texto"]) for c in campaigns.load()[:10]]
gt.append((config.DEMO_DIR / "sms_correos.png",
           "Correos: Su paquete ES2849301 está retenido en aduanas por tasas impagadas (1,99 €). "
           "Abone hoy para evitar su devolución: https://correos-aduanas.top/pago"))

local = set(ollama_server.list_local_models())
vlms = [m for m in ("qwen2.5vl:7b", "qwen2.5vl:3b") if m in local]
print("Modelos de visión disponibles:", vlms)
vlm_rows = []
for m in vlms:
    vision.read_image(gt[0][0], model=m)  # calentamiento (carga en VRAM)
    for path, text in gt:
        data, met = vision.read_image(path, model=m)
        doms_gt = set(extractors.find_domains(text))
        doms = set(extractors.find_domains(data["texto_completo"] + " " + " ".join(data["enlaces"])))
        vlm_rows.append({"modelo": m, "imagen": path.stem, "CER": jiwer.cer(norm(text), norm(data["texto_completo"])),
                         "enlace_exacto": (doms_gt <= doms) if doms_gt else np.nan, "s": met["ms"] / 1000})
vlm = pd.DataFrame(vlm_rows)
display(vlm.groupby("modelo").agg(CER_medio=("CER", "mean"), CER_mediano=("CER", "median"),
                                  enlaces_exactos=("enlace_exacto", "mean"), latencia_s=("s", "mean")).round(3))
vlm["enlace_exacto"] = vlm["enlace_exacto"].astype(float)
RESULTS["vlm"] = vlm.groupby("modelo").mean(numeric_only=True).round(4).to_dict(orient="index")

In [ ]:
g = vlm.groupby("modelo").agg(CER=("CER", "mean"), enlaces=("enlace_exacto", "mean"), s=("s", "mean"))
txt = "**Decisión.** Un carácter mal leído en un dominio (`correos-aduanas.top` frente a `correos.es`) cambia el veredicto, así que priorizamos la exactitud. "
if {"qwen2.5vl:7b", "qwen2.5vl:3b"} <= set(g.index):
    a7, a3 = g.loc["qwen2.5vl:7b"], g.loc["qwen2.5vl:3b"]
    txt += (f"El 7B obtiene un CER de {a7.CER:.3f} y copia exactamente el {a7.enlaces * 100:.0f} % de los enlaces "
            f"({a7.s:.1f} s por imagen); el 3B, {a3.CER:.3f} y {a3.enlaces * 100:.0f} % ({a3.s:.1f} s). ")
    txt += ("Nos quedamos con el **7B**." if a7.CER <= a3.CER else
            "El 3B lee igual de bien en este conjunto; mantenemos el 7B por su mejor comprensión de la escena y dejamos el 3B como opción ligera.")
elif len(g):
    a7 = g.iloc[0]
    txt += (f"Con `{g.index[0]}`: CER {a7.CER:.3f}, enlaces exactos {a7.enlaces * 100:.0f} %, {a7.s:.1f} s por imagen. "
            "Para comparar con el 3B: `python -m diputado.ai.ollama_server pull qwen2.5vl:3b` y volver a ejecutar.")
display(Markdown(txt))

## 3. Búsqueda de campañas conocidas: CLIP frente a SigLIP2 frente a e5

**Consultas.** Para cada campaña creamos una **variante** realista: cambiamos importes, dominios y hora, y
la renderizamos con **otra plantilla** (un SMS pasa a WhatsApp y viceversa). Un buen buscador debe
encontrar la campaña original aunque cambien los detalles y el aspecto.

In [ ]:
from diputado.ai import embeddings
from diputado.media import render

camps = campaigns.load()


def variant(c: dict, k: int) -> tuple[Image.Image, str]:
    t = re.sub(r"\d+,\d{2}", f"{2 + k},{(17 * k) % 100:02d}", c["texto"])
    t = re.sub(r"https?://[^\s]+", f"https://{c['id'].split('_')[0]}-{['info', 'pago', 'web'][k % 3]}.com/x{k}", t)
    if c["canal"] in ("sms", "whatsapp"):
        img = (render.render_whatsapp if c["canal"] == "sms" else render.render_sms)(c["remitente"], t, hour=f"1{k}:0{k}")
    elif c["canal"] == "email":
        img = render.render_email(c["remitente"], c["nombre"] + " (aviso)", t)
    else:
        img = render.render_web(c["nombre"], t)
    return img, t


queries = [(i, *variant(c, k)) for i, c in enumerate(camps) for k in range(2)]
q_imgs = [q[1] for q in queries]
q_txts = [q[2] for q in queries]
truth = np.array([q[0] for q in queries])
db_imgs = [config.CAMPAIGNS_DIR / "img" / f"{c['id']}.png" for c in camps]
print(f"{len(queries)} consultas contra {len(camps)} campañas")


def retrieval(S):
    ranks = np.array([(np.argsort(-S[i]) == truth[i]).nonzero()[0][0] for i in range(len(S))])
    return {"Recall@1": (ranks == 0).mean(), "Recall@3": (ranks < 3).mean(), "MRR": (1 / (ranks + 1)).mean()}


sims = {}
for kind in ("clip", "siglip"):
    t0 = time.perf_counter()
    sims[f"{kind} imagen→imagen"] = embeddings.embed_images(q_imgs, kind) @ embeddings.embed_images(db_imgs, kind).T
    sims[f"{kind} imagen→descripción"] = embeddings.embed_images(q_imgs, kind) @ embeddings.embed_image_texts([c["descripcion"] for c in camps], kind).T
    print(kind, f"{(time.perf_counter() - t0):.1f} s")
sims["e5 texto→texto"] = embeddings.embed_text(q_txts) @ embeddings.embed_text([f"{c['nombre']}. {c['texto']}" for c in camps]).T
ret = pd.DataFrame({k: retrieval(S) for k, S in sims.items()}).T
display(ret.round(3))
RESULTS["recuperacion"] = ret.round(4).to_dict(orient="index")

### Calibración de los rangos de similitud

La aplicación combina canales con similitudes en escalas muy distintas (e5 da ~0,8 incluso entre textos
no relacionados). Reescalamos cada canal a [0, 1] con `(s - bajo) / (alto - bajo)`, donde:

* **bajo** = percentil 95 de la similitud con campañas **incorrectas** (por debajo, ruido);
* **alto** = mediana de la similitud con la campaña **correcta**.

Los guardamos en `artifacts/campaign_ranges.json`, que lee `src/diputado/core/campaigns.py`.

In [ ]:
def calib(S):
    pos = S[np.arange(len(S)), truth]
    mask = np.ones_like(S, bool); mask[np.arange(len(S)), truth] = False
    neg = S[mask]
    return float(np.percentile(neg, 95)), float(np.median(pos)), pos, neg


ranges = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, (name, key) in zip(axes, [("texto", "e5 texto→texto"), ("imagen", "siglip imagen→imagen"), ("imagen_texto", "siglip imagen→descripción")]):
    lo, hi, pos, neg = calib(sims[key])
    ranges[name] = [round(lo, 4), round(max(hi, lo + 0.02), 4)]
    ax.hist(neg, bins=40, alpha=0.6, label="incorrecta", color="#9aa5b8", density=True)
    ax.hist(pos, bins=15, alpha=0.7, label="correcta", color="#d7263d", density=True)
    ax.axvline(lo, c="k", ls="--"); ax.axvline(hi, c="k", ls=":")
    ax.set_title(f"{key}\nbajo={lo:.3f} · alto={hi:.3f}", fontsize=9); ax.legend(fontsize=7)
plt.tight_layout()
(config.ARTIFACTS_DIR / "campaign_ranges.json").write_text(json.dumps(ranges, indent=2), encoding="utf-8")
campaigns._ranges.cache_clear()
print("Rangos guardados:", ranges)

**Decisión.** SigLIP2 frente a CLIP: elegimos SigLIP2 si gana en recuperación imagen→imagen, que es el
canal visual de la aplicación. Además, su pérdida sigmoide da puntuaciones imagen-texto absolutas (no
relativas a un lote), lo que lo hace más adecuado como **control de calidad** de las infografías.

## 4. Voz de salida: MMS-TTS frente a Bark

La voz del aviso debe ser, ante todo, **inteligible** para una persona mayor. Lo medimos con una prueba de
ida y vuelta: sintetizamos el texto, lo transcribimos con Whisper turbo y calculamos el WER. También
medimos el RTF.

In [ ]:
from diputado.ai import asr, bark, tts
from diputado.ai.audio_io import resample

frases = [
    "Cuidado: esto tiene todas las señales de una estafa. No pulses el enlace y no pagues nada.",
    "Tu banco nunca te pedirá el código que te llega por SMS. Cuelga y llama al número de tu tarjeta.",
    "Parece legítimo, aunque conviene mantener la precaución. Ante cualquier duda, vuelve a preguntarnos.",
    "Hemos detectado un dominio que imita a la web de Correos y una petición de pago de uno con noventa y nueve euros.",
]
import soundfile as sf

tts_rows = []
tmp = config.OUTPUTS_DIR / "nb04"
tmp.mkdir(exist_ok=True)
for name, fn in [("MMS-TTS", lambda t, i: tts.synthesize(t, seed=i)), ("Bark", lambda t, i: bark.synthesize(t, voice="v2/es_speaker_1", seed=i))]:
    fn(frases[0][:40], 0)  # calentamiento
    for i, f in enumerate(frases):
        t0 = time.perf_counter()
        audio, sr = fn(f, i)
        el = time.perf_counter() - t0
        p = tmp / f"{name}_{i}.wav"
        sf.write(p, audio, sr)
        hyp = asr.transcribe(p)[0]["text"]
        tts_rows.append({"voz": name, "WER ida y vuelta": jiwer.wer(norm(f), norm(hyp)), "RTF": el / (len(audio) / sr)})
tts_df = pd.DataFrame(tts_rows).groupby("voz").mean()
display(tts_df.round(3))
RESULTS["tts"] = tts_df.round(4).to_dict(orient="index")

In [ ]:
m_, b_ = tts_df.loc["MMS-TTS"], tts_df.loc["Bark"]
display(Markdown(
    f"**Decisión.** MMS-TTS para el aviso: WER de ida y vuelta {m_['WER ida y vuelta']:.3f} frente a "
    f"{b_['WER ida y vuelta']:.3f} de Bark, pero con RTF {m_.RTF:.2f} en CPU frente a {b_.RTF:.1f} de Bark en GPU "
    f"(unas {b_.RTF / m_.RTF:.0f} veces más rápido) y de forma determinista. Bark suena más natural (respiraciones, "
    "pausas) pero es lento y a veces inventa o se salta palabras; por eso lo usamos justo para lo contrario: "
    "**generar llamadas de estafa verosímiles** para la demo y para entrenar VozSinteticaNet."))

## 5. Infografía: pasos de SDXL-Turbo frente a calidad

SDXL-Turbo está destilado para generar en 1 a 4 pasos. Medimos la **similitud SigLIP2** entre la imagen y
su prompt (un control de calidad automático, como en la sesión 11 de IQA) y la latencia.

In [ ]:
from diputado.ai import imagegen
from diputado.media import infographic

if imagegen.enabled():
    sd_rows, grid = [], []
    prompts = [infographic.SCENES[k] for k in ("credenciales_otp", "falso_familiar", "pago_transferencia", "verde")]
    imagegen.generate(prompts[0], seed=0, steps=1)  # calentamiento
    for steps in (1, 2, 4):
        for j, pr in enumerate(prompts):
            img, m = imagegen.generate(pr, seed=j, steps=steps)
            sd_rows.append({"pasos": steps, "SigLIP2": embeddings.image_text_score(img, pr), "s": m["ms"] / 1000})
            if j < 2:
                grid.append((steps, img))
    sd = pd.DataFrame(sd_rows).groupby("pasos").agg(SigLIP2=("SigLIP2", "mean"), latencia_s=("s", "mean"))
    display(sd.round(4))
    RESULTS["sdxl"] = sd.round(4).to_dict(orient="index")
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    for col, steps in enumerate((1, 2, 4)):
        imgs = [im for s, im in grid if s == steps]
        for row in range(2):
            axes[row, col].imshow(imgs[row]); axes[row, col].axis("off")
        axes[0, col].set_title(f"{steps} paso(s)")
    plt.tight_layout()
else:
    print("Sin GPU: la infografía usa una ilustración de plantilla.")

In [ ]:
if imagegen.enabled():
    display(Markdown(
        f"**Decisión.** 2 pasos: SigLIP2 pasa de {sd.loc[1, 'SigLIP2']:.3f} (1 paso) a {sd.loc[2, 'SigLIP2']:.3f} y "
        f"con 4 pasos apenas sube a {sd.loc[4, 'SigLIP2']:.3f}. En este portátil la latencia (~{sd['latencia_s'].mean():.0f} s) "
        "casi no depende de los pasos: la domina mover la UNet de 5 GB entre RAM y VRAM (*CPU offload*), "
        "imprescindible con 8 GB porque Qwen2.5-VL y Qwen3 también necesitan la GPU. La aplicación precarga "
        f"SDXL-Turbo al arrancar y la infografía llega después del veredicto, sin bloquearlo. El umbral de calidad "
        f"({infographic.QUALITY_THRESHOLD}) queda muy por debajo de la puntuación típica, así que casi siempre basta "
        "una generación; la segunda semilla solo entra en los casos malos."))

**¿Y vídeo generativo?** Las referencias del curso usan Stable Video Diffusion. En 8 GB de VRAM, SVD-XT
necesita *offload* a CPU y tarda varios minutos por clip de 2-3 s, incompatible con una alerta que debe
llegar en segundos. Montamos el vídeo-alerta con `moviepy` (infografía con un paneo vertical suave,
narración y subtítulos): tarda segundos, el mensaje es legible y lleva la marca «Generado por IA».

## 6. Razonamiento: Qwen3 8B frente a Qwen2.5 7B

Reutilizamos el test manual del notebook 02. Qwen3 se ejecuta **sin modo de razonamiento extendido**
(`think=False`) y con salida JSON validada por esquema: el razonamiento largo multiplicaba la latencia por
5-10 sin mejorar el F1 en pruebas preliminares.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

from diputado.ai import llm

test = pd.read_csv(config.DATA_DIR / "test_manual.csv")
cache_q3 = json.loads((config.ARTIFACTS_DIR / "llm_test_manual.json").read_text(encoding="utf-8"))
llm_rows = [{"modelo": config.OLLAMA_LLM, "F1": f1_score(test["estafa"], [cache_q3[str(i)]["p"] >= 0.5 for i in range(len(test))]),
             "AUC": roc_auc_score(test["estafa"], [cache_q3[str(i)]["p"] for i in range(len(test))]),
             "latencia_s": np.mean([v["ms"] for v in cache_q3.values()]) / 1000}]
if config.OLLAMA_AGENT_LLM in local:
    orig = config.OLLAMA_LLM
    llm.config.OLLAMA_LLM = config.OLLAMA_AGENT_LLM
    ps, ms = [], []
    for _, r in test.iterrows():
        d, m = llm.analyze(r["canal"], r["texto"])
        ps.append(d["probabilidad_estafa"]); ms.append(m["ms"])
    llm.config.OLLAMA_LLM = orig
    llm_rows.append({"modelo": config.OLLAMA_AGENT_LLM, "F1": f1_score(test["estafa"], np.array(ps) >= 0.5),
                     "AUC": roc_auc_score(test["estafa"], ps), "latencia_s": np.mean(ms) / 1000})
llm_df = pd.DataFrame(llm_rows).set_index("modelo")
display(llm_df.round(3))
RESULTS["llm"] = llm_df.round(4).to_dict(orient="index")

In [ ]:
(config.ARTIFACTS_DIR / "seleccion_modelos.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("Resultados guardados en artifacts/seleccion_modelos.json")

## Resumen de decisiones

| Función | Elegido | Por qué |
|---|---|---|
| Transcripción | Whisper large-v3-turbo (GPU) / base (CPU) | Menor WER con audio telefónico y RTF muy por debajo de 1 |
| Lectura de imágenes | Qwen2.5-VL 7B | Copia exacta de enlaces y dominios, que deciden el veredicto |
| Búsqueda de campañas | SigLIP2 (imagen) + e5 (texto) | Mejor recuperación; puntuaciones absolutas útiles como IQA |
| Voz de salida | MMS-TTS | Inteligible, determinista, rápido en CPU |
| Llamadas sintéticas | Bark | Prosodia natural: el tipo de voz que queremos detectar |
| Infografía | SDXL-Turbo, 2 pasos + control SigLIP2 | Calidad similar a 4 pasos con la mitad de latencia |
| Razonamiento | Qwen3 8B sin modo *thinking* | Mejor F1 en el test manual con latencia de segundos |